# 🎯 DipRadar — Bootstrap ML (Colab)

Corre cada célula **uma de cada vez**, por ordem.
O Parquet acumula no Google Drive — podes fechar o Colab entre sessões sem perder progresso.

**Fase A — CamadaA (Price + Macro):**

| Passo | O que faz | Tempo estimado |
|-------|-----------|----------------|
| Batch 0–200 | Macro bulk fetch (3 pedidos) + backfill preços + features | ~25 min |
| Batch 200–400 | idem (macro já em cache no Drive) | ~25 min |
| Batch 400–600 | idem | ~25 min |
| Batch 600–800 | idem | ~25 min |
| Batch 800–fim | idem | ~10 min |

**Fase B — CamadaB (Fundamentais PIT):**

| Passo | O que faz | Tempo estimado |
|-------|-----------|----------------|
| Fund 0–200 | quarterly_income_stmt + balance_sheet + cashflow PIT | ~35 min |
| Fund 200–400 | idem | ~35 min |
| Fund 400–600 | idem | ~35 min |
| Fund 600–fim | idem | ~25 min |
| Treino XGBoost + KNN | train_model.py (16 features) | ~5 min |
| **Total** | | **~4 h 30 min** |

> ⚠️ **Cada sessão nova do Colab**: volta a correr as células 1, 2, 3 e 4 antes de continuar.

> ℹ️ O download macro (VIX, SPY, T10Y2Y) é feito automaticamente no **primeiro batch de cada sessão** — não há passo separado.

> 🛡️ **CamadaB usa Point-in-Time rigoroso**: reporting lag de 45 dias + fallback NaN (nunca `tk.info`). Se um batch falhar por Rate Limit, volta a correr a célula — os tickers já processados são saltados.

## 1 · Montar Google Drive

In [16]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado


## 2 · Instalar dependências

In [17]:
%pip install -q yfinance scikit-learn pyarrow fastparquet xgboost shap pandas-datareader
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 3 · Clonar / actualizar repositório

In [18]:
import os

REPO_DIR = '/content/DipRadar'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull --quiet
    print('✅ Repositório actualizado')
else:
    !git clone --quiet https://github.com/romeurf/DipRadar.git $REPO_DIR
    %cd $REPO_DIR
    print('✅ Repositório clonado')

DRIVE_DIR = '/content/drive/MyDrive/DipRadar'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'📁 Drive dir: {DRIVE_DIR}')

/content/DipRadar
✅ Repositório actualizado
📁 Drive dir: /content/drive/MyDrive/DipRadar


## 4 · Configurar API Key (Tiingo)

> Guarda a chave em: **Colab → Secrets** (ícone da chave 🔑) → Nome: `TIINGO_API_KEY`

In [19]:
import os
from google.colab import userdata

try:
    os.environ['TIINGO_API_KEY'] = userdata.get('TIINGO_API_KEY')
    print('✅ TIINGO_API_KEY carregada dos Secrets do Colab')
except Exception:
    import getpass
    os.environ['TIINGO_API_KEY'] = getpass.getpass('🔑 Cola a tua TIINGO_API_KEY: ')
    print('✅ TIINGO_API_KEY definida via input')

✅ TIINGO_API_KEY carregada dos Secrets do Colab


---
## ⚡ FASE A — CamadaA: Preços + Macro
---

## 5 · Batch 1 — tickers 0–200

> O `build_global_macro_df()` corre automaticamente no início deste batch (3 pedidos de rede, ~3 s).  
> Se a sessão expirar a meio, volta a correr — os tickers já processados são saltados.

In [20]:
!python bootstrap_ml.py \
    --layer price \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:27:35 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:27:35 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:27:35 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:27:35 [bootstrap_ml] 🔪 Slice UNIVERSE[0:200] = 200 tickers
00:27:35 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:27:37 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:27:37 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:27:37 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:27:37 [bootstrap_ml] [CamadaA] Janela: 2006-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:27:37 [bootstrap_ml] [CamadaA] 2006-05-01 → 2026-05-01 |

## 6 · Batch 2 — tickers 200–400

In [21]:
!python bootstrap_ml.py \
    --layer price \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:29:30 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:29:30 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:29:30 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:29:30 [bootstrap_ml] 🔪 Slice UNIVERSE[200:400] = 200 tickers
00:29:30 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:29:31 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:29:31 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:29:31 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:29:31 [bootstrap_ml] [CamadaA] Janela: 2006-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:29:31 [bootstrap_ml] [CamadaA] 2006-05-01 → 2026-05-01

## 7 · Batch 3 — tickers 400–600

In [22]:
!python bootstrap_ml.py \
    --layer price \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:31:24 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:31:24 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:31:24 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:31:24 [bootstrap_ml] 🔪 Slice UNIVERSE[400:600] = 200 tickers
00:31:24 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:31:24 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:31:24 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:31:24 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:31:24 [bootstrap_ml] [CamadaA] Janela: 2006-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:31:24 [bootstrap_ml] [CamadaA] 2006-05-01 → 2026-05-01

## 8 · Batch 4 — tickers 600–800

In [23]:
!python bootstrap_ml.py \
    --layer price \
    --slice 600 800 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:33:19 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:33:19 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:33:19 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:33:19 [bootstrap_ml] 🔪 Slice UNIVERSE[600:800] = 198 tickers
00:33:19 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:33:19 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:33:20 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:33:20 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:33:20 [bootstrap_ml] [CamadaA] Janela: 2006-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:33:20 [bootstrap_ml] [CamadaA] 2006-05-01 → 2026-05-01

## 9 · Batch 5 — tickers 800 até ao fim

In [24]:
!python bootstrap_ml.py \
    --layer price \
    --slice 800 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:35:09 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:35:09 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:35:09 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:35:09 [bootstrap_ml] 🔪 Slice UNIVERSE[800:999] = 0 tickers
00:35:09 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:35:10 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:35:10 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:35:10 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:35:10 [bootstrap_ml] [CamadaA] Janela: 2006-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:35:10 [bootstrap_ml] [CamadaA] 2006-05-01 → 2026-05-01 |

---
## 🧮 FASE B — CamadaB: Fundamentais Point-in-Time

> Enriquece cada alerta do Parquet com `gross_margin`, `fcf_yield`, `revenue_growth` e `de_ratio`  
> extraídos dos `quarterly_income_stmt` / `balance_sheet` / `cashflow` **já públicos na data do dip**  
> (reporting lag de 45 dias). Fallback é **NaN** — nunca `tk.info`.
---

## 10 · Fund Batch 1 — tickers 0–200 (US Core)

> ⏱️ ~35 min. Faz 3 pedidos de rede extra por ticker (quarterly statements).  
> Se o Yahoo Finance devolver Rate Limit, o Colab irá abrandar — é normal. Volta a correr se necessário.

In [25]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:36:07 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:36:07 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:36:07 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:36:07 [bootstrap_ml] 🔪 Slice UNIVERSE[0:200] = 200 tickers
00:36:07 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:36:09 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:36:09 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:36:09 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:36:09 [bootstrap_ml] [CamadaB] Janela: 2019-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:36:09 [bootstrap_ml] [CamadaB] 2019-05-01 → 2026-05-01 |

## 11 · Fund Batch 2 — tickers 200–400

In [26]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:43:50 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:43:50 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:43:50 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:43:50 [bootstrap_ml] 🔪 Slice UNIVERSE[200:400] = 200 tickers
00:43:50 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:43:51 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:43:51 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:43:51 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:43:51 [bootstrap_ml] [CamadaB] Janela: 2019-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:43:51 [bootstrap_ml] [CamadaB] 2019-05-01 → 2026-05-01

## 12 · Fund Batch 3 — tickers 400–600

In [27]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:47:10 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:47:10 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:47:10 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:47:10 [bootstrap_ml] 🔪 Slice UNIVERSE[400:600] = 200 tickers
00:47:10 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:47:11 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:47:11 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:47:11 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:47:11 [bootstrap_ml] [CamadaB] Janela: 2019-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:47:11 [bootstrap_ml] [CamadaB] 2019-05-01 → 2026-05-01

## 13 · Fund Batch 4 — tickers 600–fim (Europa / Edge Cases)

In [28]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 600 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

00:50:21 [bootstrap_ml] [universe] 798 tickers únicos (hardcoded, zero scraping)
00:50:21 [bootstrap_ml] 📋 Universo total: 798 tickers ML
00:50:21 [bootstrap_ml] 📐 Contrato: 16 features (['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike'])
00:50:21 [bootstrap_ml] 🔪 Slice UNIVERSE[600:999] = 198 tickers
00:50:21 [bootstrap_ml] [macro] Bulk-fetch VIX + SPY (2006-05-01 → 2026-05-06) ...
00:50:22 [bootstrap_ml] [macro] T10Y2Y carregado via FRED (pandas_datareader)
00:50:22 [bootstrap_ml] [macro] DataFrame macro pronto: 7306 dias | 2006-05-01 → 2026-05-01
00:50:22 [bootstrap_ml] [macro] Bulk fetch concluído — 7306 dias de história macro
00:50:22 [bootstrap_ml] [CamadaB] Janela: 2019-05-01 → 2026-05-01 | data_dir: /content/drive/MyDrive/DipRadar
00:50:22 [bootstrap_ml] [CamadaB] 2019-05-01 → 2026-05-01

In [29]:
import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/DipRadar/ml_training_fund.parquet")
print(df["outcome_label"].value_counts())
print(f"\nNaNs por feature:\n{df[['fcf_yield','revenue_growth','gross_margin']].isna().sum()}")

outcome_label
NEUTRAL    2977
WIN_20     1244
WIN_40     1105
LOSS_15     709
Name: count, dtype: int64

NaNs por feature:
fcf_yield         6007
revenue_growth    6035
gross_margin      6007
dtype: int64


---
## 🏋️ FASE C — Treino do Ensemble Completo
---

## 14 · Treino XGBoost + KNN + Platt Scaling

> Só depois de todos os batches (Fase A + Fase B) estarem completos.  
> Usa o `train_model.py` dedicado — **não** o bootstrap. Garante que o contrato de features é lido de `ml_features.py` (fonte única de verdade).  
> O modelo agora treina com **16 features** (12 price/macro + 4 fundamentais PIT).

In [30]:
# --- CÉLULA: Merge do Super-Dataset (Camada A + Camada B) ---
import pandas as pd

print("A carregar Parquets do Google Drive...")
df_price = pd.read_parquet("/content/drive/MyDrive/DipRadar/ml_training_price.parquet")
df_fund  = pd.read_parquet("/content/drive/MyDrive/DipRadar/ml_training_fund.parquet")

print(f"📊 Colunas Camada A: {list(df_price.columns)}")
print(f"📊 Colunas Camada B: {list(df_fund.columns)}")
print(f"📊 Amostras Camada A (Price+Macro): {len(df_price)}")
print(f"📊 Amostras Camada B (Fundamentais PIT): {len(df_fund)}")

# Marcar a origem de cada linha (útil para debug depois)
df_price["_source"] = "price"
df_fund["_source"]  = "fund"

# Combinar — fund fica por último para que keep='last' prefira PIT reais
df_merged = pd.concat([df_price, df_fund], ignore_index=True)

# Verificar se as colunas de dedup existem
dedup_cols = [c for c in ["date", "ticker"] if c in df_merged.columns]
if len(dedup_cols) == 2:
    antes = len(df_merged)
    df_merged = df_merged.drop_duplicates(subset=dedup_cols, keep="last")
    print(f"🔁 Deduplicação: {antes} → {len(df_merged)} alertas únicos")
else:
    print(f"⚠️  Colunas {dedup_cols} disponíveis — sem deduplicação (colunas em falta: {set(['date','ticker'])-set(dedup_cols)})")

# Remover coluna auxiliar de source antes de passar ao train_model
df_merged = df_merged.drop(columns=["_source"], errors="ignore")

print(f"🚀 Total combinado final: {len(df_merged)} alertas")
print(f"📈 Distribuição de outcomes:\n{df_merged['outcome_label'].value_counts()}")

# Verificar NaNs nas features críticas
from ml_features import FEATURE_COLUMNS
nan_summary = df_merged[FEATURE_COLUMNS].isna().sum()
nan_cols = nan_summary[nan_summary > 0]
if not nan_cols.empty:
    print(f"⚠️  NaNs detectados (serão imputados pelo train_model):\n{nan_cols}")
else:
    print("✅ Sem NaNs nas features")

merged_path = "/content/drive/MyDrive/DipRadar/ml_training_merged.parquet"
df_merged.to_parquet(merged_path)
print(f"✅ Merge concluído e guardado em: {merged_path}")

A carregar Parquets do Google Drive...
📊 Colunas Camada A: ['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike', 'symbol', 'alert_date', 'price', 'label_win', 'label_further_drop', 'outcome_label', 'return_3m', 'return_6m', 'spy_return_ref']
📊 Colunas Camada B: ['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike', 'symbol', 'alert_date', 'price', 'sector', 'market_cap_b', 'label_win', 'label_further_drop', 'outcome_label', 'return_3m', 'return_6m']
📊 Amostras Camada A (Price+Macro): 11333
📊 Amostras Camada B (Fundamentais PIT): 6035
⚠️  Colunas [] disponíveis — sem deduplicação (colunas em falta: {'date', 'tick

In [32]:
df_b = df[df["sector"].notna()].copy()
df_b["alert_date"] = pd.to_datetime(df_b["alert_date"])
df_b["year"] = df_b["alert_date"].dt.year

print("NaNs em fcf_yield por ano:")
print(df_b.groupby("year")["fcf_yield"].apply(lambda x: x.isna().sum()))
print()
print("Total por ano:")
print(df_b.groupby("year").size())

NaNs em fcf_yield por ano:
year
2019     912
2020    3501
2021     646
2022     731
2023      93
2024      76
2025      48
2026       0
Name: fcf_yield, dtype: int64

Total por ano:
year
2019     912
2020    3501
2021     646
2022     731
2023      93
2024      76
2025      75
2026       1
dtype: int64


In [33]:
# --- CÉLULA FINAL: Treino do Ensemble Completo ---
!python train_model.py \
    --parquet /content/drive/MyDrive/DipRadar/ml_training_merged.parquet \
    --output-dir /content/drive/MyDrive/DipRadar

2026-05-02 00:58:54,525 [INFO] ------------------------------------------------------------
2026-05-02 00:58:54,525 [INFO] DipRadar ML - Laboratorio de Treino
2026-05-02 00:58:54,525 [INFO] ------------------------------------------------------------
2026-05-02 00:58:54,526 [INFO] [ml_data] A carregar Parquet: /content/drive/MyDrive/DipRadar/ml_training_merged.parquet
2026-05-02 00:58:54,582 [INFO] [ml_data] Parquet: 17368 registos | colunas: ['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'revenue_growth', 'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside', 'quality_score', 'drop_pct_today', 'drawdown_52w', 'rsi_14', 'atr_ratio', 'volume_spike', 'symbol', 'alert_date', 'price', 'label_win', 'label_further_drop', 'outcome_label', 'return_3m', 'return_6m', 'spy_return_ref', 'sector', 'market_cap_b']
2026-05-02 00:58:54,582 [INFO] [ml_data] Usando 16 features do bootstrap: ['macro_score', 'vix', 'spy_drawdown_5d', 'sector_drawdown_5d', 'fcf_yield', 'r

## 15 · Validar contrato de features (anti Training-Serving Skew)

> Confirma que o `.pkl` treinado e o `ml_features.py` em produção têm **exactamente as mesmas features**, pela mesma ordem.  
> **Não saltes este passo** — é o guarda que impede um crash silencioso em produção.

In [ ]:
import pickle, pathlib
from ml_features import FEATURE_COLUMNS, N_FEATURES

pkl_path = pathlib.Path('/content/drive/MyDrive/DipRadar/dip_model_price.pkl')
assert pkl_path.exists(), f'❌ Modelo não encontrado em {pkl_path}'

with open(pkl_path, 'rb') as f:
    bundle = pickle.load(f)

model_features = list(bundle['model'].feature_names_in_)
code_features  = list(FEATURE_COLUMNS)

if model_features == code_features:
    print(f'✅ Contrato OK — {N_FEATURES} features idênticas entre modelo e ml_features.py')
    for i, f in enumerate(model_features):
        print(f'  {i+1:>2}. {f}')
else:
    missing = set(code_features) - set(model_features)
    extra   = set(model_features) - set(code_features)
    print('❌ MISMATCH DETECTADO — Training-Serving Skew activo!')
    if missing: print(f'  Faltam no modelo : {missing}')
    if extra:   print(f'  Extra no modelo  : {extra}')
    raise AssertionError('Corrige antes de fazer deploy.')

## 16 · Verificar todos os ficheiros gerados

In [ ]:
import os, pathlib, pandas as pd

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')
print('📁 Ficheiros no Drive:')
for f in sorted(drive_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>8.1f} KB')

pq = drive_dir / 'ml_training_price.parquet'
if pq.exists():
    df = pd.read_parquet(pq)
    print(f'\n📊 Parquet: {len(df):,} alertas | {df["symbol"].nunique()} tickers únicos')
    print('\nDistribuição outcome_label:')
    print(df['outcome_label'].value_counts().to_string())
    # Inspecção CamadaB: % de NaNs nos fundamentais
    fund_cols = ['gross_margin', 'fcf_yield', 'revenue_growth', 'de_ratio']
    existing = [c for c in fund_cols if c in df.columns]
    if existing:
        print('\n🧮 Cobertura dos Fundamentais PIT (% não-NaN):')
        for col in existing:
            pct = df[col].notna().mean() * 100
            print(f'  {col:<20} {pct:>5.1f}% preenchido')
    else:
        print('\n⚠️  Colunas de fundamentais ainda não encontradas — corre a Fase B primeiro.')
else:
    print('⚠️  Parquet ainda não existe')

## 17 · Deploy do modelo para o Railway

### Opção A — Railway Volume (recomendado)
O Railway tem um Volume montado em `/data`.
Descarrega o `.pkl` do Drive para a tua máquina local e faz upload via Railway CLI:
```bash
railway volume cp ./dip_model_price.pkl /data/dip_model_price.pkl
railway volume cp ./dip_knn_price.pkl   /data/dip_knn_price.pkl
```

### Opção B — Commit directo (só se < 50 MB por ficheiro)
```bash
git add dip_model_price.pkl dip_knn_price.pkl
git commit -m 'feat: add trained ML models v1'
git push
```